 ## Structured Outputs -
 * **Models can be requested to provide their reponses in a format matching t a given schema .This is useful for ensuring the output can be easily parsed and used in subsequent processing.Langchain supports multiple schema types and methods for enforcing structured output**

## Pydantic -
* **Pydantic models provide the richest feature set with field validation,descriptions,and nested structures**
  - Message output alongside parsed structure : Use **include_raw=True**

## TypedDict -
* **TypedDict provides a simple alternative using Python's built in typing,ideal when you do not need runtime validation**

## DataClass -
* **DataClass is a class typically containing mainly data,although there aren't really any restrictions.You create it using the @dataclass decorator**

In [9]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x7d88a9befd40>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7d88a98dc9e0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [24]:
from pydantic import BaseModel,Field
class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="The year in which the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The rating of movie out of 10")

In [25]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x7d88a9befd40>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7d88a98dc9e0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'descr

In [26]:
response=model_with_structure.invoke("Provide the details of movie Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

## Message output alongside parsed structure

## Nested Structure

In [27]:
from pydantic import BaseModel,Field
class Actor(BaseModel):
    name: str
    role: str
class MovieDetails(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="The year in which the movie was released")
    cast:list[Actor]
    genres:list[str]
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The rating of movie out of 10")
    budget:float | None=Field(None,description="Budget in millions USD")
    
model_with_structure=model.with_structured_output(MovieDetails)
response=model_with_structure.invoke("Provide the details of movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Balthazar')], genres=['Science Fiction', 'Action', 'Thriller'], director='Christopher Nolan', rating=8.8, budget=None)

## TypedDict

In [32]:
from typing_extensions import TypedDict,Annotated
class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str,...,"The title of the movie"]
    year: Annotated[int ,...,"The year movie was released"]
    director: Annotated[str,...,"The doirector of the movie"]
    rating: Annotated[float,...,"The movie's rating out of 10"]

model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response 

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

# Dataclasses

## Using Python Dataclasses for Structured Outputs in LangChain

LangChain allows you to use standard Python `@dataclass` objects to enforce structured schemas on model responses, acting as a lightweight alternative to Pydantic.

## Core Integration Features

Here is an unordered bulleted list:

* **Schema Generation:** LangChain automatically inspects the dataclass type hints to build the JSON schema that the LLM must follow.
* **Automatic Instantiation:** When the model responds, LangChain automatically parses the output and returns a ready-to-use instance of your dataclass.
* **Native Compatibility:** Works seamlessly across modern chat models supporting the `.with_structured_output()` method.

### Workflow Implementation Steps

Here is an ordered numbered list:

1. Define your data structure using the standard `@dataclass` decorator with strict type hints.
2. Initialize your target LangChain chat model (e.g., `ChatOpenAI` or `ChatAnthropic`).
3. Bind the structure to the model by calling `model.with_structured_output(YourDataclass)`.
4. Invoke the structured model with your prompt to receive the populated dataclass object directly.

> **Note:** Unlike Pydantic, standard Python dataclasses do not strictly validate types at runtime. If you need robust data validation and field descriptions, use Pydantic's `BaseModel` instead.

---

In [33]:
from dataclasses import dataclass
from typing import List

# 1. Define your structure using a standard Python @dataclass
@dataclass
class MovieReview:
    title: str
    rating_out_of_10: int
    pros: List[str]
    cons: List[str]
    summary: str
    
# 3. Bind the dataclass schema to the model
structured_model = model.with_structured_output(MovieReview)

# 4. Invoke the model with a prompt
prompt = "Review the 2010 movie Inception. Give it a rating, pros, cons, and a short summary."
result = structured_model.invoke(prompt)
result


{'cons': ['Complex plot may confuse some viewers',
  'Pacing lags in mid-section',
  'Ambiguous ending may divide opinions'],
 'pros': ['Innovative concept blending heist and sci-fi',
  'Stunning visual effects and action sequences',
  'Strong performances from the cast',
  'Meticulously crafted layered narrative'],
 'rating_out_of_10': 8,
 'summary': 'Inception masterfully explores the boundaries of dreams and reality through a high-stakes heist. Directed by Christopher Nolan, the film combines mind-bending concepts with breathtaking visuals, though its intricate plot demands full attention. While some may find its complexity overwhelming, the execution of dream sequences and emotional depth make it a standout modern classic.',
 'title': 'Inception'}